# 03 — EDA Multivariate: Group Differences


In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
for candidate in [REPO, REPO.parent, REPO.parent.parent]:
    if (candidate / "src" / "neuro").exists():
        REPO = candidate
        break
sys.path.insert(0, str(REPO / "src"))
import os
os.chdir(REPO)
%matplotlib inline


In [ ]:

import numpy as np
import pandas as pd
from neuro.bids import validate_bids
from neuro.features import get_schaefer_masker, extract_roi_timeseries, connectivity_matrix
from neuro import viz

report = validate_bids()
runs = report["runs"][report["runs"]["bold_exists"]].head(6)  # subset for speed
masker = get_schaefer_masker(n_rois=50)
conn_mats = []
labels = []
for _, row in runs.iterrows():
    ts = extract_roi_timeseries(row["bold_path"], masker)
    conn_mats.append(connectivity_matrix(ts))
    labels.append(f"{row['subject']}_{row['task']}")
mean_conn = np.mean(conn_mats, axis=0)
viz.plot_connectivity_heatmap(mean_conn, "Mean ROI connectivity (n=6 runs)", "03_connectivity_mean.png")


In [ ]:

# Group-level mean connectivity (MDD vs ND) — first available per subject
from collections import defaultdict
sub_conn = defaultdict(list)
for _, row in report["runs"][report["runs"]["bold_exists"]].iterrows():
    ts = extract_roi_timeseries(row["bold_path"], masker)
    sub_conn[(row["subject"], row["group_short"])].append(connectivity_matrix(ts))

mdd = [np.mean(v, axis=0) for (_, g), v in sub_conn.items() if g == "MDD"][:5]
nd = [np.mean(v, axis=0) for (_, g), v in sub_conn.items() if g == "ND"][:5]
if mdd and nd:
    diff = np.mean(mdd, axis=0) - np.mean(nd, axis=0)
    viz.plot_connectivity_heatmap(diff, "MDD − ND connectivity diff", "03_connectivity_group_diff.png")


In [ ]:

from pathlib import Path
nb_name = Path.cwd().name if False else "03_eda_multivariate"
!jupyter nbconvert --to html notebooks/03_eda_multivariate.ipynb --output-dir notebooks/html 2>/dev/null || true
